#Import


In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline
import pandas as pd

#to remove warnings when using microsoft/Phi-3.5-mini-instruct
import logging
from transformers import logging as transformers_logging

# Only show errors, hide warnings
transformers_logging.set_verbosity_error()

# Model and pipeline definition

In [ ]:
torch.cuda.is_available()

model = AutoModelForCausalLM.from_pretrained("microsoft/Phi-3.5-mini-instruct",
                                              device_map="auto",
                                              torch_dtype="auto",
                                              trust_remote_code=False)

tokenizer = AutoTokenizer.from_pretrained("microsoft/Phi-3.5-mini-instruct",
                                          trust_remote_code=True)

pipe = pipeline("text-generation",
                model=model,
                tokenizer=tokenizer,
                max_new_tokens=300,
                max_length=300,
                temperature=0.2, #range [0,1]
                top_p=1) #range [0,1], usually between .8 and 1

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/195 [00:00<?, ?it/s]

...

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

added_tokens.json:   0%|          | 0.00/306 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

# Datasets loading

In [ ]:
#paths samuel
topics_path = "/content/drive/MyDrive/Uni/aa2425/TLN/DiCaro/5/topic_lab_4.csv"
definitions_path = "/content/drive/MyDrive/Uni/aa2425/TLN/DiCaro/5/def.csv"

#general paths
#topics_path = "/content/drive/MyDrive/ ... /5/topic_lab_4.csv"
#definitions_path = "/content/drive/MyDrive/ ... /5/def.csv"

from google.colab import drive
drive.mount("/content/drive", force_remount=True)

Mounted at /content/drive


In [ ]:
topics = pd.read_csv(topics_path)
definitions = pd.read_csv(definitions_path)
definitions = definitions.drop("Unnamed: 0",axis=1).drop("Termine",axis=1)

In [ ]:
topics.head()

,Topic,Count,Name,Representation,Representative_Docs
0,-1,22961,-1_of_the_in_and,"['of', 'the', 'in', 'and', 'to', 'with', 'that...",['Clinical signs of pulmonary congestion indic...
1,0,4005,0_cancer_tumor_cells_expression,"['cancer', 'tumor', 'cells', 'expression', 'ce...",['IL-6 was significantly associated with poor ...
2,1,1066,1_asthma_allergic_airway_ige,"['asthma', 'allergic', 'airway', 'ige', 'aller...",['Omalizumab exerts inhibitory effects on airw...
3,2,987,2_pain_propofol_morphine_isoflurane,"['pain', 'propofol', 'morphine', 'isoflurane',...",['The analgesic interaction between intratheca...
4,3,833,3_mutation_gene_mutations_genetic,"['mutation', 'gene', 'mutations', 'genetic', '...",['Our results suggest that genetic variation a...


In [ ]:
definitions.head()

,P1,P2,P3,P4,P5,P6,P7,P8,P9,P10,...,P34,P35,P36,P37,P38,P39,P40,P41,P42,P43
0,Indumento per la parte inferiore del corpo umano,"Indumento per le gambe, diviso per ogni gamba ...",abito indossato sulle gambe,capo di abbigliamento per le gambe,Indumento indossato nella parte inferiore del ...,Indumento per la parte inferiore del corpo che...,Capo indossabile che copre la parte inferiore ...,indumento indossabile da una persona tipicamen...,Indumento che copre le gambe di una persona,Indumento da indossare dalla vita in giù,...,Indumento adibito ad essere indossato nella pa...,Indumento resistente e popolare,abbigliamento fatto di tessuto più o meno sint...,"indumento che si indossa dalla vita in giù, pu...",capo di abbigliamento indossabile nella parte ...,capo di abbigliamento per coprire le gambe,"Vestito, abito, maschile o femminile che copre...",Capo d'abbigliamento indossato sulla parte inf...,NaN,NaN
1,Strumento scientifico per l'osservazione di mi...,Dispositivo ottico o elettronico per ispeziona...,strumento dotato di lenti che permette di visu...,strumento per osservare da vicino cose molto p...,Strumento scientifico utilizzato per ingrandir...,Strumento che permette di ingrandire oggetti e...,dispositivo per vedere oggetti visibili solo n...,dispositivo utilizzato per l'analisi e la visi...,Strumento per osservare elementi non visibili ...,Strumento impiegato per facilitare la visualiz...,...,Strumento che consente di osservare elementi n...,strumento che ingranisce oggetti piccoli,strumento scientifico con scopo di ricerca su ...,strumento per la visione ingrandita di oggetti...,strumento scientifico per osservare oggetti no...,un dispositivo scientifico che utilizza lenti ...,strumento generalmente utilizzato a scopi di r...,Strumento utilizzato per vedere oggetti di pic...,NaN,NaN
2,Situazione potenzialmente rischiosa,Situazione in cui si teme per la propria o alt...,condizione che può causare danni alle persone ...,Evento che minaccia la sicurezza di una persona,Situazione dove é minacciata la sicurezza di u...,Situazione in cui è a rischio l’incolumità di ...,Evento in cui ci si la propria vita si sente m...,situazione che compromette la sicurezza di un ...,situazione che minaccia la sicurezza di un sog...,Situazione in cui l’incolumità della persona è...,...,Situazione in cui l'incolumità fisica o psicol...,situazione che può provocare sofferenza,situazioni in cui viene messa a rischio la vit...,situazione in cui si rischia di subire una qua...,percezione di una situazione di minaccia,situazione di rischio,situazione in cui si teme di essere esposti a ...,E' una caratteristica che definisce situazioni...,NaN,NaN
3,Strategia di ricerca nello spazio degli stati,Strategia di ricerca che permette di stimare i...,regola che permette di approssimare una soluzi...,NaN,Funzione che stima la distanza dallo stato att...,Strategia adottata durante la risoluzione di u...,Funzione per approssimare un comportamento,ipotesi assunta nella ricerca scientifica e me...,Metodologia di ricerca di fatti o verità,Metedologia per trovare soluzione migliore ris...,...,Criterio utilizzato al fine dell'ottenimento d...,scorciatoia mentale utilizzata per risolvere p...,stima che viene utilizzata per calcolare i cos...,approssimazione fatta rispetto a un insieme di...,funzione che permette di calcolare un risultat...,un insieme di metodi di ricerca che facilitano...,strategia o procedimento per ricercare element...,E' un valore di stima per difetto,NaN,NaN


# Tasks

## Labelling

### Zero-shot

In [ ]:
#[0] -> zero_shot_istruzioni
#[1] -> zero_shot_iterativo
#[2] -> few_shot_istruzioni
#[3] -> few_shot_iterativo
#[4] -> chain_of_thought
labelling_results = [[],[],[],[],[]]

**Prompting basato su istruzioni**

In [ ]:
for n in range(1,len(topics)):
  zero_shot_prompt = [{
                      "role": "system",
                      "content": "You're a topic labeller system. Given a name, representation and representative document you generate the related label."

                    },
                    {
                      "role": "user",
                      "content": f"name: {topics.loc[n]["Name"]}, representation: {topics.loc[n]["Representation"]}, representative document: {topics.loc[n]["Representative_Docs"]}.\
                                  The label must have a max of 5 words, summarizing the main representative document objectives with respect to representation words.\
                                  The output should be concise and easily interpretable.\
                                  There must be only the label as the ouput, nothing more.",
                    }
                  ]
  output = pipe(zero_shot_prompt)
  #print(output[0]["generated_text"]) #prompt + answer
  print(f"Topic: {n}, label: {output[0]["generated_text"][2]["content"]}") #just the answer
  labelling_results[0].append(output[0]["generated_text"][2]["content"])

Topic: 1, label:  Breast cancer prognosis and treatment response prediction
Topic: 2, label:  Anti-IgE therapy reduces airway inflammation in allergic asthma
Topic: 3, label:  Propofol-Analgesia-Spinal Anesthesia
Topic: 4, label:  Genetic susceptibility in metabolic syndrome and FMTC mutations
Topic: 5, label:  Endometriosis impact on pregnancy outcomes and ART pregnancy risks
Topic: 6, label:  Prognostic Lymph Node Metastasis Chemotherapy Impact
Topic: 7, label:  Retinal implant effects, ocular complications, visual adaptation, glaucoma risk, retinal recovery
Topic: 8, label:  Intestinal mucosal colitis treatment in IBD patients
Topic: 9, label:  Cardiac Ischemia Reperfusion Protection
Topic: 10, label:  miRNA expression prognostic therapeutic cancer
Topic: 11, label:  CKD-related arterial stiffness risk in coronary patients
Topic: 12, label:  HIV infection, CD4 impairment, antiretroviral management
Topic: 13, label:  Schizophrenia, Social Cognition, Genetic Links, Mitochondrial Dysfu

**Prompting iterativo**

In [ ]:
for n in range(1,len(topics)):
  zero_shot_prompt = [{
                      "role": "system",
                      "content": "You're a topic labeller system. Given a name, representation and representative document you generate the related label by following the instructions."

                    },
                    {
                      "role": "user",
                      "content": f"name: {topics.loc[n]["Name"]}, representation: {topics.loc[n]["Representation"]}, representative document: {topics.loc[n]["Representative_Docs"]}.\
                                  Provide only the label as output, nothing else.\
                                  Instructions:\
                                  1. Read the representative document and extract a brief summary of the main object.\
                                  2. Compare the words inside the extracted informations with the words inside the representation given as input and redefine the expressions using the latter ones when possible.\
                                  3. Create a label, of max 5 word, that conceptualizes such content. Such label does not have to be syntactycally correct but rather interpretable.\
                                  4. If the label if longer than 5 words discard it and get back to instruction 3.",
                    }
                  ]
  output = pipe(zero_shot_prompt)
  #print(output[0]["generated_text"]) #prompt + answer
  print(f"Topic: {n}, label: {output[0]["generated_text"][2]["content"]}") #just the answer
  labelling_results[1].append(output[0]["generated_text"][2]["content"])

Topic: 1, label:  Breast cancer prognosis markers
Topic: 2, label:  Allergic Asthma Airway Ige Treatment
Topic: 3, label:  Spinal anesthesia analgesia propofol
Topic: 4, label:  Gene Mutations Risk Assessment
Topic: 5, label:  Endometriosis Pregnancy Outcomes
Topic: 6, label:  Chemotherapy Prognosis Lymph Node Metastasis
Topic: 7, label:  Retinal Implant Complications
Topic: 8, label:  Intestinal IBD Gut Colitis
Topic: 9, label:  Cardiac Ischemia Reperfusion Protection
Topic: 10, label:  miRNA prognostic cancer therapy
Topic: 11, label:  CKD_coronary_risk_stiffness
Topic: 12, label:  HIV Surgical Management
Topic: 13, label:  Schizophrenia-related disorders and treatments
Topic: 14, label:  Knee ACL Tendon Joint OA
Topic: 15, label:  HCV treatment strategy
Topic: 16, label:  Plant TOR Functions
Topic: 17, label:  Dental implant stability and bone changes
Topic: 18, label:  LV myocardial CRT ventricular patients echocardiography
Topic: 19, label:  Care Meta-Reviews Physician Experience


### Few-shot

**Few-shot examples**

In [ ]:
example1 = {"Representative_Docs": "A 65-year-old patient arrives at the ER complaining of a sudden onset of shortness of breath and a persistent productive cough. Physical examination reveals wheezing in the lower lobes, and a history of COPD is noted in their chart.",
             "representation":      "dyspnea, wheezing, sputum, airflow, chronic, pulmonary",
             "label":               "Chronic Obstructive Pulmonary Exacerbation"}

example2 = {"Representative_Docs": "Following a successful appendectomy, the patient is monitored in the PACU. The surgical site is checked for drainage or signs of infection. The nurse administers IV analgesics for pain management and records the vital signs every four hours.",
             "representation":      "post-op, incision, sutures, recovery, sterile, analgesia",
             "label":               "Post-Surgical Care & Wound Monitoring"}

example3 = {"Representative_Docs": "A patient with a family history of Type 2 Diabetes undergoes a fasting glucose test. The results show an elevated A1c level of 7.2%. The physician discusses starting Metformin and refers the patient to a dietician for glycemic counseling.",
             "representation":      "insulin, glucose, metabolic, hyperglycemia, A1c, endocrine",
             "label":               "Diabetes Mellitus Diagnosis & Management"}

**Prompt basato su istruzioni**

In [ ]:
for n in range(1,len(topics)):
  few_shot_prompt = [{
                      "role": "system",
                      "content": "You're a topic labeller system.\
                                  Given a name, representation and representative document you generate the related label.\
                                  Use the given examples as guides to help you produce better outputs."

                    },
                    {
                      "role": "user",
                      "content": f"name: {topics.loc[n]["Name"]}, representation: {topics.loc[n]["Representation"]}, representative document: {topics.loc[n]["Representative_Docs"]}.\
                                  The label must contain max 5 words, capturing the main representative document objective with respect to representation words.\
                                  The output should be concise and easily interpretable.\
                                  There must be only the label as the ouput, nothing more.\
                                  Examples:\
                                  example1: representative document:{example1["Representative_Docs"]}, representation: {example1["representation"]}, label: {example1["label"]}.\
                                  example2: representative document:{example2["Representative_Docs"]}, representation: {example2["representation"]}, label: {example2["label"]}.\
                                  example3: representative document:{example3["Representative_Docs"]}, representation: {example3["representation"]}, label: {example3["label"]}.",
                    }
                  ]
  output = pipe(few_shot_prompt)
  #print(output[0]["generated_text"]) #prompt + answer
  print(f"Topic: {n}, label: {output[0]["generated_text"][2]["content"]}") #just the answer
  labelling_results[2].append(output[0]["generated_text"][2]["content"])

Topic: 1, label:  Breast Cancer Prognostic Markers
Topic: 2, label:  Allergic Asthma Inflammation Reduction
Topic: 3, label:  Anesthetic Pain Management
Topic: 4, label:  Genetic Variants & Metabolic Syndrome Risk
Topic: 5, label:  Pregnancy Endometriosis Complications
Topic: 6, label:  Lymph node involvement & chemotherapy timing
Topic: 7, label:  Ocular Complications & Retinal Adaptation
Topic: 8, label:  Intestinal Colitis Relapse Prediction & Treatment
Topic: 9, label:  Ischemic Heart Reperfusion Protection
Topic: 10, label:  miRNA expression, cancer therapy, prognostic biomarker, label: miRNA Therapeutic Potential.
Topic: 11, label:  CKD-related arterial stiffness
Topic: 12, label:  HIV Infection & Surgical Outcomes
Topic: 13, label:  Schizophrenia Social Cognition Impacts
Topic: 14, label:  ACL Tear & Knee Kinematics
Topic: 15, label:  Hepatitis Viral Infection & NK Cells
Topic: 16, label:  Plant TOR Functions & Drought Response
Topic: 17, label:  Implant Bone Level Changes
Topic

**Prompt iterativo**

In [ ]:
for n in range(1,len(topics)):
  few_shot_prompt = [{
                      "role": "system",
                      "content": "You're a topic labeller system.\
                                  Given a name, representation and representative document you generate the related label.\
                                  Use the given examples as guides to help you produce better outputs by following the instructions."

                    },
                    {
                      "role": "user",
                      "content": f"name: {topics.loc[n]["Name"]}, representation: {topics.loc[n]["Representation"]}, representative document: {topics.loc[n]["Representative_Docs"]}.\
                                  The label must contain max 5 words, capturing the main representative document objective with respect to representation words.\
                                  The output should be concise and easily interpretable.\
                                  There must be only the label as the ouput, nothing more.\
                                  Examples:\
                                  example1: representative document:{example1["Representative_Docs"]}, representation: {example1["representation"]}, label: {example1["label"]}.\
                                  example2: representative document:{example2["Representative_Docs"]}, representation: {example2["representation"]}, label: {example2["label"]}.\
                                  example3: representative document:{example3["Representative_Docs"]}, representation: {example3["representation"]}, label: {example3["label"]}.\
                                  Instructions:\
                                  1. Read the representative document and extract a brief summary of the main object.\
                                  2. Compare the words inside the extracted informations with the words inside the representation given as input and redefine the expressions using the latter ones when possible.\
                                  3. Create a label, of max 5 word, that conceptualizes such content. Such label does not have to be syntactycally correct but rather interpretable.\
                                  4. If the label if longer than 5 words discard it and get back to instruction 3.",
                    }
                  ]
  output = pipe(few_shot_prompt)
  #print(output[0]["generated_text"]) #prompt + answer
  print(f"Topic: {n}, label: {output[0]["generated_text"][2]["content"]}") #just the answer
  labelling_results[3].append(output[0]["generated_text"][2]["content"])

Topic: 1, label:  Breast Cancer Prognosis & Treatment
Topic: 2, label:  Allergic Asthma Airway Inflammation Omalizumab
Topic: 3, label:  Spinal Anesthesia & Analgesia Management
Topic: 4, label:  Gene Mutations & Metabolic Syndrome Risk
Topic: 5, label:  Endometriosis & Pregnancy Complications
Topic: 6, label:  Lymph node involvement & prognosis prediction
Topic: 7, label:  Retinal Complications & Surgery Recovery
Topic: 8, label:  Intestinal Colitis, IBD, Gut Health, Mucosal Repair, Anti-Colitis Effect.
Topic: 9, label:  Ischemic Heart Protection & Reperfusion
Topic: 10, label:  miRNA Therapeutics & Cancer Prognosis
Topic: 11, label:  CKD Arterial Stiffness Risk
Topic: 12, label:  HIV Infection & Surgical Outcomes
Topic: 13, label:  Schizophrenia Social Cognition Impacts
Topic: 14, label:  ACL repair & knee kinematics
Topic: 15, label:  HCV Infection & NK Activation
Topic: 16, label:  Plant TOR Functions & Drought Response
Topic: 17, label:  Implant Surface & Bone Level Changes
Topic:

### Chain-of-thoughts

In [ ]:
for n in range(1,len(topics)):
  cot_shot_prompt = [{
                      "role": "system",
                      "content": "You're a topic labeller system.\
                                  Given a name, representation and representative document you generate the related label.\
                                  Use the given examples as guides to help you produce better outputs."

                    },
                    {
                      "role": "user",
                      "content": f"name: {topics.loc[n]["Name"]}, representation: {topics.loc[n]["Representation"]}, representative document: {topics.loc[n]["Representative_Docs"]}.\
                                  The label must contain max 5 words, capturing the main representative document objective with respect to representation words.\
                                  The output should be concise and easily interpretable.\
                                  There must be only the label as the ouput, followed by a brief (max 75 words) explanation of the chain of thoughts that brought you to such conclusion.\
                                  Examples:\
                                  example1: representative document:{example1["Representative_Docs"]}, representation: {example1["representation"]}, label: {example1["label"]}.\
                                  example2: representative document:{example2["Representative_Docs"]}, representation: {example2["representation"]}, label: {example2["label"]}.\
                                  example3: representative document:{example3["Representative_Docs"]}, representation: {example3["representation"]}, label: {example3["label"]}.",
                    }
                  ]

  output = pipe(cot_shot_prompt) #(LENTO!)
  #print(output[0]["generated_text"]) #prompt + answer
  print(f"Topic: {n}, label: {output[0]["generated_text"][2]["content"]}\n------------------------------\n") #just the answer
  #labelling_results[4].append(output[0]["generated_text"][2]["content"])

Topic: 1, label:  Label: Breast Cancer Prognostic Markers

The representative documents discuss the association of specific biological markers (IL-6, COX-2, apoptosis) with the prognosis and treatment response in breast cancer patients. These markers are linked to poor prognosis, angiogenesis, lymph node metastasis, and treatment resistance. The label "Breast Cancer Prognostic Markers" succinctly captures the essence of these documents, highlighting the focus on prognostic factors in breast cancer research and management. The choice of words reflects the critical elements of the documents: "Breast Cancer" as the disease context, "Prognostic" indicating the predictive nature of the markers, and "Markers" referring to the biological indicators under study. This label effectively conveys the main objective of the documents, which is to identify and understand prognostic markers in breast cancer for improved patient outcomes.
------------------------------

Topic: 2, label:  Label: Allergi

### Output visualization

In [ ]:
#[0] -> zero_shot_istruzioni
#[1] -> zero_shot_iterativo
#[2] -> few_shot_istruzioni
#[3] -> few_shot_iterativo
#[4] -> chain_of_thought
df = pd.DataFrame({"zero_shot_istruzioni":labelling_results[0],
                   "zero_shot_iterativo":labelling_results[1],
                   "few_shot_istruzioni":labelling_results[2],
                   "few_shot_iterativo":labelling_results[3]})
                   #"chain_of_thought":labelling_results[4]})


with pd.option_context("display.max_rows", None, "display.max_columns", None, "display.max_colwidth", None):
  display(df)

,zero_shot_istruzioni,zero_shot_iterativo,few_shot_istruzioni,few_shot_iterativo
0,Breast cancer prognosis and treatment response prediction,Breast cancer prognosis markers,Breast Cancer Prognostic Markers,Breast Cancer Prognosis & Treatment
1,Anti-IgE therapy reduces airway inflammation in allergic asthma,Allergic Asthma Airway Ige Treatment,Allergic Asthma Inflammation Reduction,Allergic Asthma Airway Inflammation Omalizumab
2,Propofol-Analgesia-Spinal Anesthesia,Spinal anesthesia analgesia propofol,Anesthetic Pain Management,Spinal Anesthesia & Analgesia Management
3,Genetic susceptibility in metabolic syndrome and FMTC mutations,Gene Mutations Risk Assessment,Genetic Variants & Metabolic Syndrome Risk,Gene Mutations & Metabolic Syndrome Risk
4,Endometriosis impact on pregnancy outcomes and ART pregnancy risks,Endometriosis Pregnancy Outcomes,Pregnancy Endometriosis Complications,Endometriosis & Pregnancy Complications
5,Prognostic Lymph Node Metastasis Chemotherapy Impact,Chemotherapy Prognosis Lymph Node Metastasis,Lymph node involvement & chemotherapy timing,Lymph node involvement & prognosis prediction
6,"Retinal implant effects, ocular complications, visual adaptation, glaucoma risk, retinal recovery",Retinal Implant Complications,Ocular Complications & Retinal Adaptation,Retinal Complications & Surgery Recovery
7,Intestinal mucosal colitis treatment in IBD patients,Intestinal IBD Gut Colitis,Intestinal Colitis Relapse Prediction & Treatment,"Intestinal Colitis, IBD, Gut Health, Mucosal Repair, Anti-Colitis Effect."
8,Cardiac Ischemia Reperfusion Protection,Cardiac Ischemia Reperfusion Protection,Ischemic Heart Reperfusion Protection,Ischemic Heart Protection & Reperfusion
9,miRNA expression prognostic therapeutic cancer,miRNA prognostic cancer therapy,"miRNA expression, cancer therapy, prognostic biomarker, label: miRNA Therapeutic Potential.",miRNA Therapeutics & Cancer Prognosis


## Definitions

### Zero-shot

**Prompting basato su istruzioni**

In [ ]:
collection = []

for i in range(0,len(definitions)):
  terms_from_definitions = dict()
  cont = 1
  for definition in definitions.loc[i].dropna():
    #print(definition)
    zero_shot_prompt = [{
                      "role": "system",
                      "content": "Sei un sistema il quale, passata una definizione, restituisce la miglior parola descritta dalla definizione fornita."

                    },
                    {
                      "role": "user",
                      "content": f"definizione: {definition}.\
                                  Nell'output voglio venga prodotto solamente il termine che pensi essere descritto dalla definizione passata in input.\
                                  Tale termine deve essere esattamente una singola parola del dizionario italiano.\
                                  Non deve essere prodotto nulla oltre a quanto richiesto.",
                    }
                  ]
    #print("\n---------------------------------------\n")
    output = pipe(zero_shot_prompt)
    #print(output[0]["generated_text"]) #prompt + answer
    print(f"Definizione: {definition}\nTermine individuato:{output[0]["generated_text"][2]["content"]}\n") #just the answer
    term = output[0]["generated_text"][2]["content"].lower()
    if terms_from_definitions.get(term) == None: terms_from_definitions[term] = [f"P{cont}"]
    else: terms_from_definitions[term].append(f"P{cont}")
    cont+=1
  collection.append(terms_from_definitions)
  print("\n--------------------------PROSSIMO TERMINE--------------------------\n\n")

Definizione: Indumento per la parte inferiore del corpo umano
Termine individuato: Cravatta

Definizione: Indumento per le gambe, diviso per ogni gamba può essere lungo o corto e di diversi tessuti
Termine individuato: Calzoni

Definizione: abito indossato sulle gambe
Termine individuato: Calzoni

Definizione: capo di abbigliamento per le gambe
Termine individuato: Calzoni

Definizione: Indumento indossato nella parte inferiore del corpo, copre le gambe.
Termine individuato: Culotte

Definizione: Indumento per la parte inferiore del corpo che può coprire parzialmente o totalmente le gambe.
Termine individuato: Culotte

Definizione: Capo indossabile che copre la parte inferiore del corpo, in cui ogni gamba è coperta singolarmente
Termine individuato: Gonna

Definizione: indumento indossabile da una persona tipicamente di colore blu e particolarmente resistente
Termine individuato: Jeans

Definizione: Indumento che copre le gambe di una persona
Termine individuato: Calzoni

Definizione: 

**Output visualization**

In [ ]:
original_terms = ["pantalone","microscopio","pericolo","euristica"]

for index,group in enumerate(collection):
  print(original_terms[index])
  for term in group:
    print(f"{term}: {group[term]}")
  print("\n----------------------------------------------\n")


pantalone
 cravatta: ['P1', 'P10', 'P33', 'P37', 'P40']
 calzoni: ['P2', 'P3', 'P4', 'P9', 'P12', 'P13', 'P14', 'P21', 'P22', 'P25', 'P35']
 culotte: ['P5', 'P6', 'P11', 'P16', 'P31']
 gonna: ['P7', 'P19', 'P24', 'P27', 'P28', 'P29', 'P36', 'P38', 'P39']
 jeans: ['P8', 'P30']
 calze: ['P15', 'P32']
 calzone: ['P17']
 cintura: ['P18']
 cappuccio: ['P20']
 cappotto: ['P23']
 vestito: ['P26']
 gilet: ['P34']

----------------------------------------------

microscopio
 microscope: ['P1', 'P5', 'P14', 'P25']
 microscopio: ['P2', 'P3', 'P4', 'P6', 'P7', 'P8', 'P10', 'P11', 'P12', 'P15', 'P16', 'P18', 'P21', 'P23', 'P24', 'P27', 'P28', 'P29', 'P34', 'P35', 'P37', 'P38', 'P39']
 telescopio: ['P9', 'P19', 'P31', 'P32', 'P36']
 radar: ['P13']
 micromacchina: ['P17']
 microscoppio: ['P20']
 ottica: ['P22', 'P30']
 apparecchio: ['P26']
 ingranaggio: ['P33']

----------------------------------------------

pericolo
 rischio: ['P1', 'P15', 'P17', 'P18', 'P34', 'P36']
 pericolo: ['P2', 'P6', 'P10', 

**Prompting iterativo**

In [ ]:
collection = []

for i in range(0,len(definitions)):
  terms_from_definitions = dict()
  cont = 1
  for definition in definitions.loc[i].dropna():
    #print(definition)
    zero_shot_prompt = [{
                      "role": "system",
                      "content": "Sei un sistema il quale, passata una definizione, restituisce esclusivamente il miglior termine descritto dalla definizione fornita seguendo le istruzioni date."

                    },
                    {
                      "role": "user",
                      "content": f"definizione: {definition}.\
                                  Nell'output voglio venga prodotto solamente il termine che pensi essere descritto dalla definizione passata in input.\
                                  Tale termine deve essere esattamente una singola parola del dizionario italiano.\
                                  Non deve essere prodotto nulla oltre al singolo.\
                                  Istruzioni:\
                                  1. A partire dalla definizione risali a 5 termini descritti da essa.\
                                  2. Crea una definizione poco più generale rispetto a quella fornita ed estrai 5 termini a partire da essa.\
                                  3. Crea una definizione poco più specifica rispetto a quella fornita ed estrai 5 termini a partire da essa.\
                                  4. Dall'insieme complessivo di termini ottenuto restituisci il termine che li accomuna semanticamente.\
                                  4. Restituisci esclusivamente il singolo termine individuato. Non voglio che tu restituisca le definizioni o ragionamenti che hai fatto.",
                    }
                  ]
    #print("\n---------------------------------------\n")
    output = pipe(zero_shot_prompt)
    #print(output[0]["generated_text"]) #prompt + answer
    print(f"Definizione: {definition}\nTermine individuato:{output[0]["generated_text"][2]["content"]}\n") #just the answer
    term = output[0]["generated_text"][2]["content"].lower()
    if terms_from_definitions.get(term) == None: terms_from_definitions[term] = [f"P{cont}"]
    else: terms_from_definitions[term].append(f"P{cont}")
    cont+=1
  collection.append(terms_from_definitions)
  print("\n--------------------------PROSSIMO TERMINE--------------------------\n\n")

Definizione: Indumento per la parte inferiore del corpo umano
Termine individuato: Pantaloni

Definizione: Indumento per le gambe, diviso per ogni gamba può essere lungo o corto e di diversi tessuti
Termine individuato: Calzatura

Definizione: abito indossato sulle gambe
Termine individuato: Calzoni

Definizione: capo di abbigliamento per le gambe
Termine individuato: Calzatura

Definizione: Indumento indossato nella parte inferiore del corpo, copre le gambe.
Termine individuato: Calzini

Definizione: Indumento per la parte inferiore del corpo che può coprire parzialmente o totalmente le gambe.
Termine individuato: Calzini

Definizione: Capo indossabile che copre la parte inferiore del corpo, in cui ogni gamba è coperta singolarmente
Termine individuato: Gonna

Definizione: indumento indossabile da una persona tipicamente di colore blu e particolarmente resistente
Termine individuato: Jeans

Definizione: Indumento che copre le gambe di una persona
Termine individuato: Calze

Definizion

**Output visualization**

In [ ]:
original_terms = ["pantalone","microscopio","pericolo","euristica"]

for index,group in enumerate(collection):
  print(original_terms[index])
  for term in group:
    print(f"{term}: {group[term]}")
  print("\n----------------------------------------------\n")


pantalone
 pantaloni: ['P1', 'P5', 'P11', 'P16', 'P23']
 calzatura: ['P2', 'P3', 'P4', 'P6', 'P9', 'P12', 'P13', 'P14', 'P15', 'P17', 'P21', 'P22', 'P24', 'P25', 'P27', 'P28', 'P29', 'P32', 'P33', 'P35', 'P37', 'P39', 'P40']
 gonna: ['P7']
 jeans: ['P8', 'P20']
 vestito: ['P10', 'P19']
 abbigliamento: ['P18', 'P34']
 abito: ['P26', 'P36']
 calzoni: ['P30', 'P31']
 cappotto: ['P38']

----------------------------------------------

microscopio
 microscoppia: ['P1', 'P4']
 microscopio: ['P2', 'P3', 'P6', 'P7', 'P8', 'P10', 'P14', 'P15', 'P16', 'P21', 'P22', 'P23', 'P24', 'P28', 'P29', 'P30', 'P34']
 microscope: ['P5']
 telescopio: ['P9', 'P19', 'P25', 'P31', 'P32', 'P36']
 occhiature: ['P11']
 microscoppio: ['P12']
 radiografia: ['P13']
 micronutrienti: ['P17']
 ottica: ['P18']
 occhio: ['P20', 'P27', 'P37', 'P39']
 tecnologia: ['P26']
 ingranaggio: ['P33']
 micromicroscopio: ['P35']
 occhiali: ['P38']

----------------------------------------------

pericolo
 rischio: ['P1', 'P17', 'P18'

### Few-shot

**Few-shot examples**

In [ ]:
example1 = {"definition1": "Un insieme di suoni e melodie che ci piace ascoltare per emozionarci o ballare.",
            "definition2": "L'arte di organizzare suoni e silenzi nel tempo, seguendo criteri di armonia, melodia e ritmo",
            "definition3": "Un fenomeno acustico complesso basato sulla manipolazione intenzionale di frequenze d'onda, ampiezze e timbri all'interno di una struttura sintattica definita da intervalli e rapporti matematici.",
            "term": "Musica"}

example2 = {"definition1": "Qualcosa che si sfoglia per leggere storie o imparare cose nuove.",
            "definition2": "Una raccolta di fogli stampati e rilegati insieme, protetti da una copertina, che contiene un testo o delle immagini.",
            "definition3":  "Un supporto informativo strutturato in segnature e destinato alla conservazione e alla trasmissione di contenuti intellettuali attraverso la codifica testuale.",
            "term": "Libro"}

**Prompt basato su istruzioni**

In [ ]:
collection = []

for i in range(0,len(definitions)):
  terms_from_definitions = dict()
  cont = 1
  for definition in definitions.loc[i].dropna():
    #print(definition)
    few_shot_prompt = [{
                      "role": "system",
                      "content": "Sei un sistema il quale, passata una definizione, restituisce la miglior parola descritta dalla definizione fornita.\
                                  Usa come riferimento gli esempi per comprendere come, a partire da definizioni, si possano ottenere i termini associati.\
                                  I termini associati agli esempi non sono per forza quelli che dovrai assegnare alle definizioni che ti verranno passate, ma solo delle spiegazioni su come siano ottenuti a partire dalle loro definizioni."

                    },
                    {
                      "role": "user",
                      "content": f"definizione: {definition}.\
                                  Nell'output voglio venga prodotto solamente il termine che pensi essere descritto dalla definizione passata in input.\
                                  Tale termine deve essere esattamente una singola parola del dizionario italiano.\
                                  Non deve essere prodotto nulla oltre al singolo termine richiesto.\
                                  Le etichette associate agli esempi non sono da intendere come quelle che devi assegnare alle definizioni passate, ma solo come guida.\
                                  Esempi:\
                                  Esempio1: definizione1:{example1["definition1"]}, definizione2: {example1["definition2"]}, definizione3: {example1["definition3"]} termine: {example1["term"]}.\
                                  Esempio2: definizione1:{example2["definition1"]}, definizione2: {example2["definition2"]}, definizione3: {example2["definition3"]} termine: {example2["term"]}.",
                    }
                  ]
    #print("\n---------------------------------------\n")
    output = pipe(few_shot_prompt)
    #print(output[0]["generated_text"]) #prompt + answer
    print(f"Definizione: {definition}\nTermine individuato:{output[0]["generated_text"][2]["content"]}\n") #just the answer
    term = output[0]["generated_text"][2]["content"].lower()
    if terms_from_definitions.get(term) == None: terms_from_definitions[term] = [f"P{cont}"]
    else: terms_from_definitions[term].append(f"P{cont}")
    cont+=1
  collection.append(terms_from_definitions)
  print("\n--------------------------PROSSIMO TERMINE--------------------------\n\n")

Definizione: Indumento per la parte inferiore del corpo umano
Termine individuato: Manica

Definizione: Indumento per le gambe, diviso per ogni gamba può essere lungo o corto e di diversi tessuti
Termine individuato: Indumento per le gambe

Definizione: abito indossato sulle gambe
Termine individuato: Abito

Definizione: capo di abbigliamento per le gambe
Termine individuato: Abito

Definizione: Indumento indossato nella parte inferiore del corpo, copre le gambe.
Termine individuato: Pantaloni

Definizione: Indumento per la parte inferiore del corpo che può coprire parzialmente o totalmente le gambe.
Termine individuato: Abito

Definizione: Capo indossabile che copre la parte inferiore del corpo, in cui ogni gamba è coperta singolarmente
Termine individuato: Scarpa

Definizione: indumento indossabile da una persona tipicamente di colore blu e particolarmente resistente
Termine individuato: Manufatto

Definizione: Indumento che copre le gambe di una persona
Termine individuato: Manica



**Output visualization**

In [ ]:
original_terms = ["pantalone","microscopio","pericolo","euristica"]

for index,group in enumerate(collection):
  print(original_terms[index])
  for term in group:
    print(f"{term}: {group[term]}")
  print("\n----------------------------------------------\n")

pantalone
 manica: ['P1', 'P9', 'P11']
 indumento per le gambe: ['P2']
 abito: ['P3', 'P4', 'P6', 'P13', 'P16', 'P17', 'P19', 'P20', 'P22', 'P25', 'P27', 'P30', 'P33', 'P35', 'P37', 'P38', 'P39', 'P40']
 pantaloni: ['P5']
 scarpa: ['P7']
 manufatto: ['P8', 'P34']
 manzo: ['P10', 'P23', 'P24']
 indumento: ['P12', 'P31', 'P36']
 indumento per coprire le gambe: calzatura.: ['P14', 'P28']
 vestiario da indossare sulle gambe: calzini.: ['P15']
 abbbigliamento: ['P18']
 indumento per le gambe: calzatura.: ['P21']
 libro: ['P26']
 indumento che copre le gambe: calzatura.: ['P29']
 vestiario per la parte le gambe: calzini.: ['P32']

----------------------------------------------

microscopio
 libro: ['P1', 'P3', 'P4', 'P5', 'P6', 'P7', 'P8', 'P9', 'P10', 'P11', 'P12', 'P13', 'P14', 'P15', 'P18', 'P19', 'P20', 'P21', 'P23', 'P26', 'P28', 'P31', 'P32', 'P33', 'P35', 'P36', 'P37', 'P38', 'P39']
 manifattura: ['P2']
 manoscritto: ['P16', 'P24', 'P25', 'P29', 'P34']
 manipolatore di microscopi: ['P

**Prompt iterativo**

In [ ]:
collection = []

for i in range(0,len(definitions)):
  terms_from_definitions = dict()
  cont = 1
  for definition in definitions.loc[i].dropna():
    #print(definition)
    few_shot_prompt = [{
                      "role": "system",
                      "content": "Sei un sistema il quale, passata una definizione, restituisce esclusivamente il miglior termine descritto dalla definizione fornita seguendo le istruzioni date.\
                                  Usa come riferimento gli esempi per comprendere come ottenere il termine associato passate definizioni.\
                                  I termini associati agli esempi non sono per forza quelli che dovrai assegnare alle definizioni che ti verranno passate, ma solo delle spiegazioni su come siano ottenuti a partire dalle loro definizioni."

                    },
                    {
                      "role": "user",
                      "content": f"definizione: {definition}.\
                                  Nell'output voglio venga prodotto solamente il termine che pensi essere descritto dalla definizione passata in input.\
                                  Tale termine deve essere esattamente una singola parola del dizionario italiano.\
                                  Non deve essere prodotto nulla oltre al singolo termine richiesto.\
                                  Le etichette associate agli esempi non sono da intendere come quelle che devi assegnare alle definizioni passate, ma solo come guida.\
                                  Istruzioni:\
                                  1. A partire dalla definizione risali a 5 termini descritti da essa.\
                                  2. Crea una definizione poco più generale rispetto a quella fornita ed estrai 5 termini a partire da essa.\
                                  3. Crea una definizione poco più specifica rispetto a quella fornita ed estrai 5 termini a partire da essa.\
                                  4. Dall'insieme complessivo di termini ottenuto restituisci il termine che li accomuna semanticamente.\
                                  5. Restituisci esclusivamente il singolo termine individuato. Non voglio che tu restituisca le definizioni o ragionamenti che hai fatto.\
                                  Esempi:\
                                  Esempio1: definizione1:{example1["definition1"]}, definizione2: {example1["definition2"]}, definizione3: {example1["definition3"]} termine: {example1["term"]}.\
                                  Esempio2: definizione1:{example2["definition1"]}, definizione2: {example2["definition2"]}, definizione3: {example2["definition3"]} termine: {example2["term"]}.",
                    }
                  ]
    #print("\n---------------------------------------\n")
    output = pipe(few_shot_prompt)
    #print(output[0]["generated_text"]) #prompt + answer
    print(f"Definizione: {definition}\nTermine individuato:{output[0]["generated_text"][2]["content"]}\n") #just the answer
    term = output[0]["generated_text"][2]["content"].lower()
    if terms_from_definitions.get(term) == None: terms_from_definitions[term] = [f"P{cont}"]
    else: terms_from_definitions[term].append(f"P{cont}")
    cont+=1
  collection.append(terms_from_definitions)
  print("\n--------------------------PROSSIMO TERMINE--------------------------\n\n")

Definizione: Indumento per la parte inferiore del corpo umano
Termine individuato: Libro

Definizione: Indumento per le gambe, diviso per ogni gamba può essere lungo o corto e di diversi tessuti
Termine individuato: Manzo

Definizione: abito indossato sulle gambe
Termine individuato: Libro.

Definizione: capo di abbigliamento per le gambe
Termine individuato: Manoscritto.

Definizione: Indumento indossato nella parte inferiore del corpo, copre le gambe.
Termine individuato: Manzo

Definizione: Indumento per la parte inferiore del corpo che può coprire parzialmente o totalmente le gambe.
Termine individuato: Manzo

Definizione: Capo indossabile che copre la parte inferiore del corpo, in cui ogni gamba è coperta singolarmente
Termine individuato: Manoscritto.

Definizione: indumento indossabile da una persona tipicamente di colore blu e particolarmente resistente
Termine individuato: Manufatto.

Definizione: Indumento che copre le gambe di una persona
Termine individuato: Manoscritto.

D

**Output visualization**

In [ ]:
original_terms = ["pantalone","microscopio","pericolo","euristica"]

for index,group in enumerate(collection):
  print(original_terms[index])
  for term in group:
    print(f"{term}: {group[term]}")
  print("\n----------------------------------------------\n")

pantalone
 libro: ['P1', 'P21']
 manzo: ['P2', 'P5', 'P6', 'P12', 'P13', 'P33', 'P35', 'P39', 'P40']
 libro.: ['P3', 'P20', 'P24', 'P25', 'P26']
 manoscritto.: ['P4', 'P7', 'P9', 'P10', 'P11', 'P14', 'P22', 'P27', 'P36', 'P38']
 manufatto.: ['P8', 'P34']
 manipolazione delle onde sonore.: ['P15', 'P17', 'P28', 'P29']
 manzo.: ['P16', 'P23', 'P37']
 abbigliamento.: ['P18']
 manipolazione di frequenze d'onda, ampiezze e timbri.: ['P19']
 manica

1. definizione generale: una parte di un indumento che copre una parte del corpo., definizione2: struttura di supporto per il tessuto in un capo di abbigliamento., definizione3: elemento di un indumento che si estende dalla parte superiore alla parte inferiore., definizione4: sezione di un indumento progettata per proteggere o coprire una parte del corpo., definizione5: elemento di un indumento che si estende dalla spalla alla gamba.

2. definizione più specifica: una sezione di un indumento che copre le gambe., definizione2: struttura di support